# Pick images from the "new" category

This notebook reads a **work directory produced by `increment`**, loads its plan, and lists the pictures the pipeline decided to add (`apply == "new"`). For each one it shows a thumbnail and basic metadata (capture time, GPS, camera).

Run `increment` first:

```text
python repair_library.py increment --library-work work --dir INBOX --work work-increment --max-hamming 12
```

Then point this notebook at that directory's `plan.json`.

Notes:
- The `work` directory in this repository is a full-collect run and has no `apply` markers, so the "new" filter matches nothing there. Use a directory that came from `increment`. If you point it at a full-collect run, the notebook still loads and lets you browse every cluster — it just won't restrict to "new".
- Thumbnails are decoded from the source images on disk (EXIF orientation applied). No photos are moved.
- Metadata is re-read from the selected image with ExifTool, so it reflects that file's own timestamps and GPS.

In [ ]:
from pathlib import Path
import json

%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from PIL import Image, ImageOps

import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "photo_repair").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from photo_repair.metadata import capture_of, gps_of, index_metadata, read_metadata

# Point this at the directory containing plan.json from the increment command.
WORK = ROOT / "work"
if not (WORK / "plan.json").is_file():
    raise SystemExit(f"no plan.json in {WORK}; run increment and set WORK accordingly")

THUMBNAIL_SIZE = (280, 220)
NEW = None
PLANS = None
MODE = None
PLICK = None

WORK

## Load the plan

Detect the format and select the "new" category. Each row is one cluster; for an increment run the `apply` field is `new`, `renamed`, `updated`, or `untouched`.

In [ ]:
plans_path = WORK / "plan.json"
PLANS = json.loads(plans_path.read_text())

if any("apply" in plan for plan in PLANS):
    MODE = "increment"
    NEW = [plan for plan in PLANS if plan.get("apply") == "new"]
    print(f"mode: increment — {len(NEW)} pictures in the 'new' category")
else:
    MODE = "full-collect"
    NEW = list(PLANS)
    print(f"mode: full-collect — no 'new' markers; browsing all {len(NEW)} clusters")

NEW

## Browse the new category

Set `PLICK` to the index of the image below, then run the next cell for its detail. The rows of the table match the grid order, so `PLICK = 4` opens the fifth thumbnail. If nothing was found, `NEW` is empty and the grid is skipped.

In [ ]:
def make_grid(plans):
    columns = min(5, max(2, len(plans)))
    rows = (len(plans) + columns - 1) // columns
    figure, axes = plt.subplots(rows, columns, figsize=(3.6 * columns, 3.0 * rows), squeeze=False)
    for axis, plan in zip(axes.flat, plans):
        try:
            with Image.open(Path(plan["pixel_source"])) as image:
                preview = ImageOps.exif_transpose(image).convert("RGB")
                preview.thumbnail(THUMBNAIL_SIZE)
                axis.imshow(preview)
        except Exception as error:
            axis.text(0.5, 0.5, str(error), ha="center", va="center", wrap=True)
        axis.set_title(f"{plan['group']}: {plan['stored_name']}\napply={plan.get('apply')}", fontsize=9)
        axis.axis("off")
    for axis in axes.flat[len(plans):]:
        axis.axis("off")
    figure.suptitle("new category", fontsize=14)
    figure.tight_layout()
    plt.show()

NEW_TABLE = pd.DataFrame([
    {
        "group": plan["group"],
        "apply": plan.get("apply"),
        "stored_name": plan["stored_name"],
        "capture_time": plan.get("capture_time"),
        "conflicts": "; ".join(plan.get("conflicts", [])),
        "members": len(plan.get("members", [])),
    }
    for plan in NEW
])
NEW_TABLE

In [ ]:
if NEW:
    make_grid(NEW)
else:
    print("no 'new' pictures in this plan")

## Detail for the selected image

`PLICK` is the index into `NEW` (same order as the grid and the table). Set it to the image you picked, then run this cell.

In [ ]:
def show_detail(plan, plans=PLANS):
    pick = int(PLICK)
    if not 0 <= pick < len(NEW):
        print("set PLICK to a valid index")
        return
    plan = NEW[pick]
    path = Path(plan["pixel_source"])

    figure, axis = plt.subplots(squeeze=False)
    try:
        with Image.open(path) as image:
            preview = ImageOps.exif_transpose(image).convert("RGB")
            preview.thumbnail(THUMBNAIL_SIZE)
            axis.imshow(preview)
    except Exception as error:
        axis.text(0.5, 0.5, str(error), ha="center", va="center", wrap=True)
    axis.set_title(plan["stored_name"], fontsize=11)
    axis.axis("off")
    figure.tight_layout()
    plt.show()

    rows = [{
        "group": plan["group"],
        "apply": plan.get("apply"),
        "stored_name": plan["stored_name"],
        "plan_capture_time": plan.get("capture_time"),
        "members": len(plan.get("members", [])),
        "max_hamming": plan.get("max_hamming", 0),
    }]
    try:
        raw = read_metadata("exiftool", [str(path)])
        meta = index_metadata(raw, [str(path)])[str(path)]
        capture, cap_key = capture_of(meta)
        lat, lon = gps_of(meta)
        rows[0].update({
            "exif_capture_time": capture.strftime("%Y:%m:%d %H:%M:%S") if capture else None,
            "capture_field": cap_key,
            "lat": round(lat, 6) if lat is not None else None,
            "lon": round(lon, 6) if lon is not None else None,
            "camera": (
                " ".join(str(meta.get(key) or "").strip() for key in ("EXIF:Make", "EXIF:Model", "EXIF:LensModel"))
                .strip("- ")
            ) or "(none)",
            "focal_length": str(meta.get("EXIF:FocalLength") or ""),
            "fnumber": str(meta.get("EXIF:FNumber") or ""),
            "exposure": str(meta.get("EXIF:ExposureTime") or ""),
            "iso": str(meta.get("EXIF:ISOSpeedRatings") or ""),
            "format": Image.open(path).format,
            "size_bytes": plan.get("size", 0),
        })
    except Exception as error:
        rows[0]["exif_error"] = str(error)

    display(pd.DataFrame(rows))

PLICK = None
show_detail(NEW[0])